In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

# Astra connection (use forward slashes on Windows)
# BUNDLE_PATH = "C:/Users/avyaa/secure-connect-mxed-benchmarking.zip"
# TOKEN = 
# KEYSPACE = "ecommerce"

# auth = PlainTextAuthProvider('token', TOKEN)
# cluster = Cluster(cloud={'secure_connect_bundle': BUNDLE_PATH}, auth_provider=auth)
# session = cluster.connect(KEYSPACE)

# DB_SYSTEM = "AstraDB"
# SERVER_VERSION = "Cassandra 4.0.11"
# DRIVER = "cassandra-driver-3.x"
# CONNECTION_PARAMS = "Astra secure bundle"
# DB_NAME = "Mxed Benchmarking"

print("Connected:", KEYSPACE)

Connected: ecommerce


In [2]:
import os, csv, json, uuid, random, time
from datetime import datetime, timedelta
from decimal import Decimal
from tqdm import tqdm

os.makedirs("./logs", exist_ok=True)
RUN_ID = str(uuid.uuid4())

CSV_PATH = "./logs/throughput.csv"
CSV_HEADER = [
    "timestamp_utc", "run_id", "db_system", "server_version", "driver",
    "connection_params", "db_name", "sequence_number", "operation_id",
    "operation_type", "execution_ms", "row_count_or_rows_affected",
    "success", "error_code", "error_message", "params_json"
]

with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerow(CSV_HEADER)

def U(s):
    import uuid
    try: return uuid.UUID(s)
    except: return uuid.uuid4()

def T(s):
    try: return datetime.fromisoformat(str(s).replace('T',' '))
    except: return datetime.now()

def D(s):
    try: return Decimal(str(s))
    except: return Decimal('0.00')

def I(s):
    try: return int(s)
    except: return 0

def B(s):
    return str(s).lower() in ('true','1','yes')

def S(s):
    return s if s and str(s).strip() else ""

def log_op(seq, op_id, op_type, exec_ms, count_or_affected, success, err_code, err_msg, params):
    row = [
        datetime.utcnow().isoformat() + "Z", RUN_ID, DB_SYSTEM, SERVER_VERSION, DRIVER,
        CONNECTION_PARAMS, DB_NAME, seq, op_id, op_type, round(exec_ms, 3),
        count_or_affected, bool(success), err_code, err_msg, json.dumps(params, default=str)
    ]
    with open(CSV_PATH, 'a', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow(row)

In [3]:
# Seed for reproducible sampling
random.seed(42)

# Small but sufficient pools; adjust LIMIT up/down as needed
product_rows = list(session.execute("SELECT id, category_id, price, name_lc FROM products LIMIT 2000"))
customer_rows = list(session.execute("SELECT id, email_lc FROM customers LIMIT 2000"))
order_rows = list(session.execute("SELECT id, customer_id, order_date FROM orders LIMIT 2000"))

PRODUCT_IDS = [r.id for r in product_rows]
PRODUCT_CATS = [r.category_id for r in product_rows if getattr(r, "category_id", None)]
PRODUCT_PRICES = [r.price for r in product_rows if getattr(r, "price", None) is not None]
CUSTOMER_IDS = [r.id for r in customer_rows]
ORDER_IDS = [r.id for r in order_rows]
CATEGORY_IDS = list({r.category_id for r in product_rows if getattr(r, "category_id", None)})

# Fallbacks if any pool is too small
if not PRODUCT_IDS:
    PRODUCT_IDS = [uuid.uuid4()]
if not CUSTOMER_IDS:
    CUSTOMER_IDS = [uuid.uuid4()]
if not CATEGORY_IDS:
    CATEGORY_IDS = [uuid.uuid4()]
if not PRODUCT_PRICES:
    PRODUCT_PRICES = [Decimal("10.00"), Decimal("20.00"), Decimal("50.00")]

print(f"Pools → products:{len(PRODUCT_IDS)} customers:{len(CUSTOMER_IDS)} categories:{len(CATEGORY_IDS)} orders:{len(ORDER_IDS)}")

Pools → products:2000 customers:2000 categories:50 orders:2000


In [4]:
# READS
stmt_r1_by_id  = session.prepare("SELECT * FROM products WHERE id = ? LIMIT 1")
stmt_r1_by_sku = session.prepare("SELECT * FROM products WHERE sku_lc = ? LIMIT 1")

# Note: Cassandra 2i supports equality; for range filters we rely on ALLOW FILTERING with an indexed predicate
stmt_r4_cat_price = session.prepare(
    "SELECT id, name, price, popularity FROM products WHERE category_id = ? AND price >= ? AND price <= ? ALLOW FILTERING"
)

# Top products within category; sort popularity client-side
stmt_r6_cat_sample = session.prepare(
    "SELECT id, name, popularity FROM products WHERE category_id = ? LIMIT 200 ALLOW FILTERING"
)

# Order history by customer; sort client-side by order_date desc
stmt_r5_orders_by_customer = session.prepare(
    "SELECT id, order_date, total_amount FROM orders WHERE customer_id = ? LIMIT 200 ALLOW FILTERING"
)

# Recent orders window; ALLOW FILTERING with order_date range
stmt_r3_recent_orders = session.prepare(
    "SELECT id, order_date FROM orders WHERE order_date >= ? AND order_date <= ? ALLOW FILTERING"
)

# Customer profile by id
stmt_r2_customer = session.prepare("SELECT * FROM customers WHERE id = ? LIMIT 1")

# WRITES
stmt_w2_cart_order = session.prepare("""
INSERT INTO orders (id, customer_id, order_date, status, payment_status, currency,
                    subtotal_amount, tax_amount, shipping_fee, discount_amount,
                    total_amount, coupon_code, shipping_address, billing_address,
                    created_at, updated_at)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

stmt_w2_cart_item = session.prepare("""
INSERT INTO order_items (id, order_id, product_id, quantity, unit_price,
                         discount_amount, total_price, created_at)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")

stmt_w3_update_qty = session.prepare("UPDATE order_items SET quantity = ?, total_price = ? WHERE id = ?")

stmt_w4_order = stmt_w2_cart_order  # same table/shape; status differs
stmt_w4_item  = stmt_w2_cart_item

stmt_w1_register = session.prepare("""
INSERT INTO customers (id, first_name, last_name, email, email_lc, phone,
                       address_line1, address_line2, city, state, country,
                       postal_code, is_active, created_at, updated_at)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

stmt_w5_profile = session.prepare("UPDATE customers SET first_name = ?, phone = ?, updated_at = ? WHERE id = ?")

In [5]:
def pick_price_band():
    p = random.choice(PRODUCT_PRICES)
    width = Decimal(str(random.choice([10, 25, 50, 75])))
    lo = max(Decimal("0.00"), p - width)
    hi = p + width
    return (lo, hi)

def r1_product_lookup():
    # 70% by id, 30% by sku_lc (simulate SKU path by reusing name_lc hash)
    use_id = random.random() < 0.7
    params = {}
    start = time.perf_counter()
    try:
        if use_id:
            pid = random.choice(PRODUCT_IDS)
            params = {"id": str(pid), "mode": "id"}
            rs = session.execute(stmt_r1_by_id, [pid])
        else:
            # fake a sku-like token from a product name_lc to keep query realistic but safe
            token = "sku-" + uuid.uuid4().hex[:8]
            params = {"sku_lc": token, "mode": "sku_lc"}
            rs = session.execute(stmt_r1_by_sku, [token])
        rows = list(rs)
        dur = (time.perf_counter() - start) * 1000
        return dur, len(rows), True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

def r4_search_filter():
    cat = random.choice(CATEGORY_IDS)
    lo, hi = pick_price_band()
    params = {"category_id": str(cat), "price_lo": float(lo), "price_hi": float(hi)}
    start = time.perf_counter()
    try:
        rs = session.execute(stmt_r4_cat_price, [cat, lo, hi])
        rows = list(rs)
        # emulate popularity sort client-side when present
        rows_sorted = sorted(rows, key=lambda r: getattr(r, "popularity", 0), reverse=True)[:20]
        dur = (time.perf_counter() - start) * 1000
        return dur, len(rows_sorted), True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

def r6_top_products():
    cat = random.choice(CATEGORY_IDS)
    params = {"category_id": str(cat), "limit": 10}
    start = time.perf_counter()
    try:
        rs = session.execute(stmt_r6_cat_sample, [cat])
        rows = list(rs)
        top = sorted(rows, key=lambda r: getattr(r, "popularity", 0), reverse=True)[:10]
        dur = (time.perf_counter() - start) * 1000
        return dur, len(top), True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

def r5_order_history():
    cust = random.choice(CUSTOMER_IDS)
    params = {"customer_id": str(cust)}
    start = time.perf_counter()
    try:
        rs = session.execute(stmt_r5_orders_by_customer, [cust])
        rows = list(rs)
        rows_sorted = sorted(rows, key=lambda r: getattr(r, "order_date", datetime.min), reverse=True)[:50]
        dur = (time.perf_counter() - start) * 1000
        return dur, len(rows_sorted), True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

def r3_recent_orders_range():
    end = datetime.utcnow()
    start_ts = end - timedelta(days=random.choice([3, 7, 14]))
    params = {"start": start_ts.isoformat()+"Z", "end": end.isoformat()+"Z"}
    start = time.perf_counter()
    try:
        rs = session.execute(stmt_r3_recent_orders, [start_ts, end])
        rows = list(rs)[:100]
        dur = (time.perf_counter() - start) * 1000
        return dur, len(rows), True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

def r2_customer_profile():
    cust = random.choice(CUSTOMER_IDS)
    params = {"customer_id": str(cust)}
    start = time.perf_counter()
    try:
        rs = session.execute(stmt_r2_customer, [cust])
        rows = list(rs)
        dur = (time.perf_counter() - start) * 1000
        return dur, len(rows), True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

# Session caches for cross-op dependencies
CART_ITEMS = []   # tuples of (order_item_id, order_id, product_id, qty, price)
REG_POOL   = []   # registered customer ids from W1

def w2_add_to_cart():
    order_id = uuid.uuid4()
    customer_id = random.choice(CUSTOMER_IDS)
    pid = random.choice(PRODUCT_IDS)
    qty = random.randint(1, 3)
    price = Decimal(str(random.choice(PRODUCT_PRICES))) if PRODUCT_PRICES else Decimal("25.00")
    now = datetime.utcnow()

    params = {"order_id": str(order_id), "customer_id": str(customer_id), "product_id": str(pid), "qty": qty, "price": float(price)}
    start = time.perf_counter()
    try:
        # create order header as cart
        session.execute(stmt_w2_cart_order, [
            order_id, customer_id, now, "cart", "pending", "USD",
            price*qty, price*qty*Decimal("0.08"), Decimal("0.00"), Decimal("0.00"),
            price*qty*Decimal("1.08"), "", "{}", "{}", now, now
        ])
        # add single line item
        oi_id = uuid.uuid4()
        session.execute(stmt_w2_cart_item, [
            oi_id, order_id, pid, qty, price, Decimal("0.00"), price*qty, now
        ])
        CART_ITEMS.append((oi_id, order_id, pid, qty, price))
        dur = (time.perf_counter() - start) * 1000
        return dur, 2, True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

def w3_update_cart_qty():
    # Ensure there is something to update
    if not CART_ITEMS:
        _ = w2_add_to_cart()
    if not CART_ITEMS:
        return 0.0, 0, False, "EmptyCart", "No cart items to update", {}

    oi_id, order_id, pid, qty, price = random.choice(CART_ITEMS)
    new_qty = max(1, qty + random.choice([-1, 1, 2]))
    params = {"order_item_id": str(oi_id), "new_qty": new_qty}
    start = time.perf_counter()
    try:
        session.execute(stmt_w3_update_qty, [new_qty, price*new_qty, oi_id])
        # Update cache
        CART_ITEMS[:] = [(i if i[0] != oi_id else (oi_id, order_id, pid, new_qty, price)) for i in CART_ITEMS]
        dur = (time.perf_counter() - start) * 1000
        return dur, 1, True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

def w4_place_order():
    order_id = uuid.uuid4()
    customer_id = random.choice(CUSTOMER_IDS)
    num_items = random.randint(2, 6)
    now = datetime.utcnow()

    # Build items first for totals
    items = []
    subtotal = Decimal("0.00")
    for _ in range(num_items):
        pid = random.choice(PRODUCT_IDS)
        qty = random.randint(1, 3)
        price = Decimal(str(random.choice(PRODUCT_PRICES))) if PRODUCT_PRICES else Decimal("25.00")
        subtotal += price*qty
        items.append((pid, qty, price))

    tax = subtotal * Decimal("0.08")
    shipping = Decimal("9.99")
    total = subtotal + tax + shipping

    params = {"order_id": str(order_id), "customer_id": str(customer_id), "num_items": num_items, "subtotal": float(subtotal)}
    start = time.perf_counter()
    try:
        session.execute(stmt_w4_order, [
            order_id, customer_id, now, "placed", "paid", "USD",
            subtotal, tax, shipping, Decimal("0.00"), total, "", "{}", "{}",
            now, now
        ])
        affected = 1
        for (pid, qty, price) in items:
            oi_id = uuid.uuid4()
            session.execute(stmt_w4_item, [
                oi_id, order_id, pid, qty, price, Decimal("0.00"), price*qty, now
            ])
            affected += 1
        dur = (time.perf_counter() - start) * 1000
        return dur, affected, True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

def w1_registration():
    cust_id = uuid.uuid4()
    email = f"mix_{uuid.uuid4().hex[:10]}@bench.local"
    phone = f"+1{random.randint(2000000000, 9999999999)}"
    now = datetime.utcnow()
    params = {"customer_id": str(cust_id), "email": email}
    start = time.perf_counter()
    try:
        session.execute(stmt_w1_register, [
            cust_id, "Mix", "User", email, email.lower(), phone,
            "123 Mixed St", "", "City", "State", "USA", "12345",
            True, now, now
        ])
        REG_POOL.append(cust_id)
        dur = (time.perf_counter() - start) * 1000
        return dur, 1, True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

def w5_update_profile():
    # Prefer recently registered; else baseline pool
    cust_id = random.choice(REG_POOL) if REG_POOL else random.choice(CUSTOMER_IDS)
    first = random.choice(["Alex","Sam","Taylor","Dev"])
    phone = f"+1{random.randint(2000000000, 9999999999)}"
    now = datetime.utcnow()
    params = {"customer_id": str(cust_id), "first_name": first}
    start = time.perf_counter()
    try:
        session.execute(stmt_w5_profile, [first, phone, now, cust_id])
        dur = (time.perf_counter() - start) * 1000
        return dur, 1, True, None, None, params
    except Exception as e:
        dur = (time.perf_counter() - start) * 1000
        return dur, 0, False, type(e).__name__, str(e)[:200], params

In [6]:
TOTAL_OPS = 500  # adjust as needed

# Weights (sum reads 65, writes 35)
weights = {
    "R1": 20,  # Product Lookup
    "R4": 22,  # Search + Filter
    "R6": 8,   # Top Products
    "R5": 7,   # Order History
    "R3": 4,   # Recent Orders
    "R2": 4,   # Customer Profile
    "W2": 12,  # Add to Cart
    "W3": 8,   # Update Cart Qty
    "W4": 10,  # Place Order
    "W1": 2,   # Registration
    "W5": 3    # Update Profile
}

registry = {
    "R1": ("read",  r1_product_lookup),
    "R4": ("read",  r4_search_filter),
    "R6": ("read",  r6_top_products),
    "R5": ("read",  r5_order_history),
    "R3": ("read",  r3_recent_orders_range),
    "R2": ("read",  r2_customer_profile),
    "W2": ("write", w2_add_to_cart),
    "W3": ("write", w3_update_cart_qty),
    "W4": ("write", w4_place_order),
    "W1": ("write", w1_registration),
    "W5": ("write", w5_update_profile)
}

# Expand plan by weight
plan = []
for op_id, pct in weights.items():
    count = round(TOTAL_OPS * (pct / 100.0))
    plan.extend([op_id] * count)

# Trim or pad to exact TOTAL_OPS using the most common read (R4)
if len(plan) > TOTAL_OPS:
    plan = plan[:TOTAL_OPS]
elif len(plan) < TOTAL_OPS:
    plan.extend(["R4"] * (TOTAL_OPS - len(plan)))

random.shuffle(plan)
print("Plan length:", len(plan), "ops")

# Execute
reads_done = writes_done = 0
t0 = time.perf_counter()

for seq, op_id in enumerate(tqdm(plan, desc="Mixed throughput", unit="op", ncols=100), start=1):
    op_type, func = registry[op_id]
    dur, count_or_aff, ok, err_code, err_msg, params = func()
    log_op(seq, op_id, op_type, dur, count_or_aff, ok, err_code, err_msg, params)
    if op_type == "read" and ok:
        reads_done += 1
    if op_type == "write" and ok:
        writes_done += 1

elapsed_ms = (time.perf_counter() - t0) * 1000.0
elapsed_s = elapsed_ms / 1000.0
qps = reads_done / elapsed_s if elapsed_s > 0 else 0.0
tps = writes_done / elapsed_s if elapsed_s > 0 else 0.0

Plan length: 500 ops


Mixed throughput: 100%|███████████████████████████████████████████| 500/500 [02:37<00:00,  3.17op/s]


In [7]:
summary_params = {
    "reads": reads_done,
    "writes": writes_done,
    "QPS": round(qps, 3),
    "TPS": round(tps, 3)
}
log_op(len(plan)+1, "SUMMARY", "mixed", elapsed_ms, len(plan), True, None, None, summary_params)

print("\n=== Mixed Phase Summary ===")
print(f"Total ops: {len(plan)}")
print(f"Elapsed: {elapsed_ms:.1f} ms")
print(f"Reads OK: {reads_done} | Writes OK: {writes_done}")
print(f"QPS: {qps:.3f} | TPS: {tps:.3f}")
print(f"CSV: {CSV_PATH}")


=== Mixed Phase Summary ===
Total ops: 500
Elapsed: 157667.0 ms
Reads OK: 325 | Writes OK: 175
QPS: 2.061 | TPS: 1.110
CSV: ./logs/throughput.csv
